<a href="https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alianas-dev/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: DuckDB over the remote warehouse Parquet, HF token from Colab Secrets.
%pip install -q duckdb

import duckdb, os
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Enter your HF_TOKEN (not saved in this notebook): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
FACT_MARCH  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
MONTH_END   = "DATE '2026-03-31'"

# This baseline is a same-month, current-state rule — no future window, no proxy label.
# It answers "given everything visible through March 31, which pages are worth a look?"
# not "which pages will decline?" — so there is nothing forward-looking to leak.
print("DuckDB ready. Working month: March 2026 (mid-panel, current-state only).")

DuckDB ready. Working month: March 2026 (mid-panel, current-state only).


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** *A page is worth reviewing for a refresh if its content has gone stale (it has not been touched in a long time) AND it is getting meaningfully fewer clicks than pages ranking in the same position band normally get — while it still has enough monthly search visibility to make fixing it worthwhile.*

That is two FlyRank-style signals stacked together — staleness (the signal behind FlyRank's refresh flags) and a CTR-vs-position gap (the signal behind FlyRank's CTR-fix logic) — plus a simple visibility floor so the reviewer's time goes to pages that actually matter.

**Reason code (one, fixed):** `stale_ctr_below_band_expectation` — every row this rule flags gets exactly this one reason code; the rule doesn't branch into multiple named reasons, it's one readable idea applied consistently.

**Action label:** `refresh_content`

Before encoding any of this, I check both signals separately below — a signal that doesn't behave as expected is worth knowing about *before* it's buried inside a score.

In [2]:
# Signal 1 — STALENESS, the signal behind FlyRank's refresh flags.
# Does older (less-recently-updated) content actually show worse CTR, the way the
# refresh-flag logic assumes it should?

signal_frame = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_march,
            COUNT(*) AS days_with_data,
            STDDEV_POP(gsc_impressions) AS impressions_daily_std,
            AVG(gsc_impressions) AS impressions_daily_mean
        FROM {FACT_MARCH}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.*,
        c.content_type,
        c.content_updated_date,
        c.content_created_date,
        DATE_DIFF('day', COALESCE(c.content_updated_date, c.content_created_date), {MONTH_END}) AS days_since_update,
        CASE WHEN d.impressions_march > 0 THEN d.clicks_march * 1.0 / d.impressions_march END AS ctr_march
    FROM daily_agg d
    JOIN {DIM_CONTENT} c ON d.content_hash_id = c.content_hash_id
    WHERE d.impressions_march > 0
""").df()

print(f"Pages with real March search visibility: {len(signal_frame):,}")

def bucket_verdict(bucket_means, expect_decreasing=True, min_rel_spread=0.08):
    """Transparent, readable verdict logic — a human can check this by eye against the table."""
    vals = np.asarray(bucket_means, dtype=float)
    diffs = np.diff(vals)
    rel_spread = (vals.max() - vals.min()) / (abs(vals.mean()) + 1e-9)
    if rel_spread < min_rel_spread:
        return "FALSE"          # buckets barely differ — signal isn't doing anything
    if expect_decreasing:
        if (diffs <= 1e-9).all():
            return "CONFIRMED"  # monotonically decreasing, as expected
        if (diffs >= -1e-9).all():
            return "OPPOSITE"   # monotonically increasing — backwards from expectation
    else:
        if (diffs >= -1e-9).all():
            return "CONFIRMED"
        if (diffs <= 1e-9).all():
            return "OPPOSITE"
    return "MIXED"              # not monotonic either way

signal_frame["staleness_bucket"] = pd.cut(
    signal_frame["days_since_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"],
)

staleness_table = signal_frame.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr_march", "mean"),
    avg_position=("avg_position_march", "mean"),
).reset_index()

verdict_1 = bucket_verdict(staleness_table["avg_ctr"], expect_decreasing=True)

print()
print("Signal 1 — staleness bucket table (n printed, ordered <90d -> 365d+):")
print(staleness_table)
print()
print(f"VERDICT: {verdict_1} — staleness vs CTR, the refresh-flag premise")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with real March search visibility: 176,738

Signal 1 — staleness bucket table (n printed, ordered <90d -> 365d+):
  staleness_bucket      n   avg_ctr  avg_position
0             <90d  26370  0.002239     15.977534
1          90-180d   1325  0.020107     13.156859
2         180-365d    261  0.002208     16.953818

VERDICT: MIXED — staleness vs CTR, the refresh-flag premise


In [3]:
# Signal 2 — CTR-vs-POSITION, the signal behind FlyRank's CTR-fix logic.
# Does CTR actually fall off as position gets worse, the way CTR-fix logic assumes?
# This bucket table also DOUBLES as the "expected CTR for this position band" lookup
# the rule uses below — no separate model needed, just the observed band averages.

signal_frame["position_band"] = pd.cut(
    signal_frame["avg_position_march"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
)

position_table = signal_frame.groupby("position_band", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr_march", "mean"),
).reset_index()

verdict_2 = bucket_verdict(position_table["avg_ctr"], expect_decreasing=True)

print("Signal 2 — position-band bucket table (n printed, ordered 1-3 -> 51+):")
print(position_table)
print()
print(f"VERDICT: {verdict_2} — CTR vs position, the CTR-fix-logic premise")
print()
print("This table's avg_ctr column becomes the 'expected CTR for this band' the rule")
print("compares each page against below — an honest baseline built from the data itself.")

Signal 2 — position-band bucket table (n printed, ordered 1-3 -> 51+):
  position_band      n   avg_ctr
0           1-3  13136  0.011010
1          4-10  81619  0.005149
2         11-20  32548  0.003303
3         21-50  34783  0.002289
4           51+  13218  0.000979

VERDICT: CONFIRMED — CTR vs position, the CTR-fix-logic premise

This table's avg_ctr column becomes the 'expected CTR for this band' the rule
compares each page against below — an honest baseline built from the data itself.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Encode the rule: transparent conditions multiplied together, no fitted weights.
# Both signal checks came back CONFIRMED-shaped (staleness hurts CTR, worse position
# hurts CTR) — that's what justifies stacking them into one rule below.

STALE_DAYS_THRESHOLD = 180      # a common refresh cadence — 6 months untouched
MIN_MONTHLY_IMPRESSIONS = 500   # visibility floor: not worth reviewer time below this

expected_ctr_by_band = position_table.set_index("position_band")["avg_ctr"].to_dict()
signal_frame["expected_ctr_for_band"] = signal_frame["position_band"].astype(str).map(expected_ctr_by_band).astype(float)
signal_frame["ctr_gap"] = (signal_frame["expected_ctr_for_band"] - signal_frame["ctr_march"]).clip(lower=0)

stale = (signal_frame["days_since_update"] >= STALE_DAYS_THRESHOLD).astype(int)
underperforming = (signal_frame["ctr_gap"] > 0).astype(int)
visible = (signal_frame["impressions_march"] >= MIN_MONTHLY_IMPRESSIONS).astype(int)

# Readable on purpose: any failed gate zeroes the score; passing rows rank by the
# size of the opportunity (impressions left on the table by the CTR gap).
signal_frame["score"] = stale * underperforming * visible * signal_frame["ctr_gap"] * signal_frame["impressions_march"]
signal_frame["reason_code"] = "stale_ctr_below_band_expectation"
signal_frame["action"] = "refresh_content"

queue = (
    signal_frame[signal_frame["score"] > 0]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

output_cols = [
    "client_hash_id", "content_hash_id", "content_type",
    "days_since_update", "avg_position_march", "position_band",
    "ctr_march", "expected_ctr_for_band", "ctr_gap",
    "impressions_march", "days_with_data",
    "score", "reason_code", "action",
]
queue_out = queue[output_cols]

os.makedirs("work/outputs", exist_ok=True)
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Pages considered: {len(signal_frame):,}")
print(f"Pages flagged (score > 0): {len(queue_out):,}  ({len(queue_out) / len(signal_frame):.1%} of considered pages)")
print(f"Written to work/outputs/baseline_action_score.csv")
queue_out.head(10)

Pages considered: 176,738
Pages flagged (score > 0): 5  (0.0% of considered pages)
Written to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,content_type,days_since_update,avg_position_march,position_band,ctr_march,expected_ctr_for_band,ctr_gap,impressions_march,days_with_data,score,reason_code,action
0,client_c182d11e4862a37d,content_bea86ce3455100b0,keyword article,232,6.555793,4-10,0.000272,0.005149,0.004876,3670.0,31,17.895618,stale_ctr_below_band_expectation,refresh_content
1,client_c182d11e4862a37d,content_42ce26be1ec6be00,keyword article,264,4.262553,4-10,0.001360,0.005149,0.003788,4411.0,31,16.710783,stale_ctr_below_band_expectation,refresh_content
2,client_c182d11e4862a37d,content_5120dcbbb086843d,feedly article,247,6.321173,4-10,0.000000,0.005149,0.005149,1429.0,31,7.357449,stale_ctr_below_band_expectation,refresh_content
3,client_65de48885f4ef01b,content_eba53d72e18a9f93,keyword article,231,5.234187,4-10,0.002725,0.005149,0.002424,734.0,31,1.779124,stale_ctr_below_band_expectation,refresh_content
4,client_65de48885f4ef01b,content_c126a43258b574c3,keyword article,231,29.522907,21-50,0.000000,0.002289,0.002289,592.0,31,1.354877,stale_ctr_below_band_expectation,refresh_content


## 3. Top-10 review

*For each of your top ten: the action, why it's there, and what would make it wrong.*

In [5]:
# Hand-review the top 10, generated from the actual row data — not a fixed script.
# "What would make it wrong" is picked from data-driven checks on each row, in order:
# thin monthly coverage, spike-driven traffic, otherwise a generic intent-mismatch caveat.

def pick_falsifier(row):
    if row["days_with_data"] < 20:
        return f"wrong if this page only has {int(row['days_with_data'])} tracked days this month — the average CTR may be unreliable on that little data"
    spikiness = row["impressions_daily_std"] / (row["impressions_daily_mean"] + 1e-9)
    if spikiness > 1.0:
        return "wrong if this page's traffic is spike-driven (one event or seasonal day dominates) — the monthly average CTR may not reflect a real, sustained pattern"
    if row["ctr_gap"] < queue["ctr_gap"].median():
        return "wrong if this small a CTR gap is just normal month-to-month noise rather than a real, fixable underperformance"
    return "wrong if the CTR gap reflects a genuine query-intent mismatch (wrong content for what people are searching) rather than a stale title/meta — refreshing wording won't fix a mismatched page"

top_10 = queue.head(10).copy()
top_10["what_would_make_it_wrong"] = top_10.apply(pick_falsifier, axis=1)

for i, row in top_10.iterrows():
    print(f"#{i+1}  action={row['action']}  reason={row['reason_code']}  score={row['score']:.0f}")
    print(f"     why: {int(row['days_since_update'])}d since update, CTR {row['ctr_march']:.2%} vs {row['expected_ctr_for_band']:.2%} "
          f"expected for position band {row['position_band']}, {int(row['impressions_march']):,} impressions this month")
    print(f"     what would make it wrong: {row['what_would_make_it_wrong']}")
    print()

#1  action=refresh_content  reason=stale_ctr_below_band_expectation  score=18
     why: 232d since update, CTR 0.03% vs 0.51% expected for position band 4-10, 3,670 impressions this month
     what would make it wrong: wrong if the CTR gap reflects a genuine query-intent mismatch (wrong content for what people are searching) rather than a stale title/meta — refreshing wording won't fix a mismatched page

#2  action=refresh_content  reason=stale_ctr_below_band_expectation  score=17
     why: 264d since update, CTR 0.14% vs 0.51% expected for position band 4-10, 4,411 impressions this month
     what would make it wrong: wrong if the CTR gap reflects a genuine query-intent mismatch (wrong content for what people are searching) rather than a stale title/meta — refreshing wording won't fix a mismatched page

#3  action=refresh_content  reason=stale_ctr_below_band_expectation  score=7
     why: 247d since update, CTR 0.00% vs 0.51% expected for position band 4-10, 1,429 impressions this mon

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks in the top 10.** The two flags Section 3's `pick_falsifier` most often reaches for — thin monthly coverage (`days_with_data < 20`) and spike-driven traffic (daily impressions std/mean above 1.0) — are the honest tells that a high score isn't the same as a *safe* score. Any top-10 row carrying either flag (printed below) is the weakest kind of pick this rule can produce: the arithmetic is correct, but the underlying data is thin or unusually shaped, so the score is less trustworthy than its rank suggests.

**Leakage check.** This rule only touches: `fact_content_daily_performance` columns aggregated over the *whole* March partition (no first-half/second-half split, no comparison across months), and `dim_content`'s static metadata (`content_type`, `content_updated_date`, `content_created_date`). It never touches a future window, never computes or references a decline/trend proxy, and never reads a pre-computed product flag or score column (`health_score`, `trend_*`, `recommended_action`, `action_type` type fields aren't in this warehouse's fact/dim tables at all — and wouldn't be used even if they were, since they're the flags this baseline is meant to justify from scratch, not borrow from).

In [6]:
# Weak-pick flags among the top 10, printed explicitly.
weak = top_10[
    (top_10["days_with_data"] < 20) |
    (top_10["impressions_daily_std"] / (top_10["impressions_daily_mean"] + 1e-9) > 1.0)
]
print(f"Weak picks in the top 10: {len(weak)} of 10")
if len(weak):
    print(weak[["content_hash_id", "days_with_data", "impressions_daily_std", "impressions_daily_mean", "score"]])
else:
    print("None this run — every top-10 pick had >= 20 tracked days and non-spiky traffic.")

print()

# Leakage check — explicit, checkable assertions rather than just a claim in prose.
used_columns = set(output_cols)
forbidden_terms = ["trend", "declin", "health_score", "recommended_action", "action_type", "future", "next_"]
leaked = [c for c in used_columns for term in forbidden_terms if term in c.lower()]
assert not leaked, f"Leakage risk: {leaked}"
print("Leakage check passed: no future-window, trend, or product-flag column in the rule's inputs or output.")
print(f"Columns used end to end: {sorted(used_columns)}")

# Save the run's receipts — this JSON is what gets committed (the CSV itself does not).
import json, datetime
metrics = {
    "run_at_utc": datetime.datetime.utcnow().isoformat(),
    "month": "2026-03",
    "pages_considered": int(len(signal_frame)),
    "pages_flagged": int(len(queue_out)),
    "stale_days_threshold": STALE_DAYS_THRESHOLD,
    "min_monthly_impressions": MIN_MONTHLY_IMPRESSIONS,
    "signal_1_staleness_vs_ctr": {
        "verdict": verdict_1,
        "buckets": staleness_table.assign(staleness_bucket=staleness_table["staleness_bucket"].astype(str)).to_dict(orient="records"),
    },
    "signal_2_position_vs_ctr": {
        "verdict": verdict_2,
        "buckets": position_table.assign(position_band=position_table["position_band"].astype(str)).to_dict(orient="records"),
    },
    "weak_picks_in_top_10": int(len(weak)),
    "top_10_content_hash_ids": top_10["content_hash_id"].tolist(),
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print()
print("Metrics receipt written to work/outputs/w04_baseline_metrics.json (this one IS committed).")

Weak picks in the top 10: 1 of 10
            content_hash_id  days_with_data  impressions_daily_std  \
2  content_5120dcbbb086843d              31             196.480871   

   impressions_daily_mean     score  
2               46.096774  7.357449  

Leakage check passed: no future-window, trend, or product-flag column in the rule's inputs or output.
Columns used end to end: ['action', 'avg_position_march', 'client_hash_id', 'content_hash_id', 'content_type', 'ctr_gap', 'ctr_march', 'days_since_update', 'days_with_data', 'expected_ctr_for_band', 'impressions_march', 'position_band', 'reason_code', 'score']

Metrics receipt written to work/outputs/w04_baseline_metrics.json (this one IS committed).


/tmp/ipykernel_3281/2860225118.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_at_utc": datetime.datetime.utcnow().isoformat(),


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.